In [ ]:
%%capture
# Installs Unsloth, Xformers (Flash Attention) and all other packages!
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import torch

In [ ]:
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
max_sequence_length= 2048
dtype= None
load_in_4bit= True

In [ ]:
# Importing Model and Tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Choose from simple models like 'unsloth/tinyllama' or 'unsloth/llama-2-7b'
    max_seq_length=max_sequence_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.8.18: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


In [ ]:
# Model Architecture
FastLanguageModel.for_inference(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=3072, out_features=3072, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=3072, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Ll

In [ ]:
# Tokenizer
text= "I love using unsloth in colab."

In [ ]:
tokens= tokenizer.tokenize(text)
print(f"Tokens: {tokens}")

Tokens: ['I', 'Ġlove', 'Ġusing', 'Ġuns', 'loth', 'Ġin', 'Ġcol', 'ab', '.']


In [ ]:
input_ids= tokenizer.encode(text)
print(f"IDs: {input_ids}")

IDs: [128000, 40, 3021, 1701, 7120, 70652, 304, 1400, 370, 13]


In [ ]:
decoded= tokenizer.decode(input_ids)
decoded

'<|begin_of_text|>I love using unsloth in colab.'

In [ ]:
print(tokenizer.special_tokens_map)

{'bos_token': '<|begin_of_text|>', 'eos_token': '<|eot_id|>', 'pad_token': '<|finetune_right_pad_id|>'}


In [ ]:
print(tokenizer.chat_template)

{{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif %}
{%- if not date_string is defined %}
    {%- if strftime_now is defined %}
        {%- set date_string = strftime_now("%d %b %Y") %}
    {%- else %}
        {%- set date_string = "26 Jul 2024" %}
    {%- endif %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}

{#- This block extracts the system message, so we can slot it into the right place. #}
{%- if messages[0]['role'] == 'system' %}
    {%- set system_message = messages[0]['content']|trim %}
    {%- set messages = messages[1:] %}
{%- else %}
    {%- set system_message = "" %}
{%- endif %}

{#- System message #}
{{- "<|start_header_id|>system<|end_header_id|>\n\n" }}
{%- if tools is not none %}
    {{- "Environment: ipython\n" }}
{%- endif %}
{{- "Cutting Knowledge Date: December 2023\n" }}
{{- 

In [ ]:
messages= [
    {"role": "system","content": "You are a helpful assistant."},
    {"role": "user","content": "Who won the world series in 2020?"},
    {"role": "assistant","content": "The Los Angeles Dodgers won the World Series in 2020."},
    {"role": "user","content": "Where was it played?"}
]

In [ ]:
inputs= tokenizer.apply_chat_template(
    messages,
    tokenize= True,
    add_generation_prompt= True,
    return_tensors= "pt"
).to("cuda")

In [ ]:
tokenizer.decode(inputs)

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 16 Aug 2026\n\nYou are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWho won the world series in 2020?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nThe Los Angeles Dodgers won the World Series in 2020.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nWhere was it played?<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n']

In [ ]:
response= model.generate(
    input_ids= inputs,
    max_new_tokens= 128,
    max_length= 500,
    use_cache= True,
    temperature= 1,
    min_p= 0.1
)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=128) and `max_length`(=500) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
response

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,    845,   5033,    220,   2366,     21,    271,   2675,    527,
            264,  11190,  18328,     13, 128009, 128006,    882, 128007,    271,
          15546,   2834,    279,   1917,   4101,    304,    220,   2366,     15,
             30, 128009, 128006,  78191, 128007,    271,    791,   9853,  12167,
          56567,   2834,    279,   4435,  11378,    304,    220,   2366,     15,
             13, 128009, 128006,    882, 128007,    271,   9241,    574,    433,
           6476,     30, 128009, 128006,  78191, 128007,    271,    791,    220,
           2366,     15,   4435,  11378,    574,   6476,   1990,    279,   9853,
          12167,  56567,    323,    279,  33225,   9332,  80775,     13,    578,
           4101,   3952,   2035,    520,  41910,   9601,   8771,    304,  59796,
             11,   8421,    

In [ ]:
result= tokenizer.batch_decode(response)

In [ ]:
print(result[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 16 Aug 2026

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Who won the world series in 2020?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The Los Angeles Dodgers won the World Series in 2020.<|eot_id|><|start_header_id|>user<|end_header_id|>

Where was it played?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The 2020 World Series was played between the Los Angeles Dodgers and the Tampa Bay Rays. The series took place at Globe Life Field in Arlington, Texas, but the Dodgers were actually hosting their World Series games at Globe Life Field since they were the host team for the series (it was the Dodgers' turn to host the World Series, as home-field advantage went to the team with the better regular season record, and the Dodgers had a better record than the Rays).<|eot_id|>


In [ ]:
model= FastLanguageModel.get_peft_model(
    model,
    r= 16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha= 16,
    lora_dropout= 0,
    bias= "none",
    use_gradient_checkpointing= True,
    random_state= 42,
    use_rslora= False,
    loftq_config= None
)

Unsloth 2026.8.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
import json
file_path= "/content/customer_service_classification_500.jsonl"

raw_data= []

with open(file_path, "r", encoding= "utf-8") as f:
  for line in f:
    raw_data.append(line)

print(raw_data[0])

{"input": "I never received the account confirmation email for our company.", "output": "SUPPORT_TEAM"}



In [ ]:
from datasets import Dataset

In [ ]:
import json
dataset= Dataset.from_list([json.loads(line) for line in raw_data])

In [ ]:
type(dataset)

datasets.arrow_dataset.Dataset

In [ ]:
dataset[10]

{'input': 'I need to add another administrator to our workspace.',
 'output': 'ACCOUNT_TEAM'}

In [ ]:
system_prompt= "You are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)"

In [ ]:
def formating_prompts(example):
  convos= []
  texts= []

  system_prompt= "You are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)"

  for input_text, output_text in zip(example["input"], example["output"]):
    conversation= [
        {"role": "system","content": system_prompt},
        {"role": "user","content": input_text},
        {"role": "assistant","content": output_text}
    ]

    text= tokenizer.apply_chat_template(conversation, tokenize= False, add_generation_prompt= True)

    texts.append(text)
  return {"text": texts}

In [ ]:
dataset= dataset.map(formating_prompts, batched= True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
dataset[0]['text']

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 16 Aug 2026\n\nYou are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI never received the account confirmation email for our company.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nSUPPORT_TEAM<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
from unsloth import is_bfloat16_supported

In [ ]:
trainer= SFTTrainer(
    model= model,
    tokenizer= tokenizer,
    train_dataset= dataset,
    dataset_text_field= "text",
    max_seq_length= max_sequence_length,
    dataset_num_proc= 2,
    packing= False,
    args= TrainingArguments(
        per_device_train_batch_size= 2,
        gradient_accumulation_steps= 4,
        warmup_steps= 5,
        max_steps= 60,
        learning_rate= 2e-4,
        fp16= not is_bfloat16_supported(),
        bf16= is_bfloat16_supported(),
        logging_steps= 1,
        optim= "adamw_8bit",
        weight_decay= 0.01,
        lr_scheduler_type= "linear",
        seed= 3407,
        output_dir= "outputs"
    )
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
trainer_stats= trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 500 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.933306
2,4.918539
3,4.730954
4,4.180838
5,3.737517
6,3.396830
7,2.874114
8,2.504122
9,2.157298
10,1.823803


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [ ]:
# Model After Fine Tuning (lora) layer
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
input= "We are looking to buy licence for our entire engineering department."
messages= [
    {"role": "system", "content":"You are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)"},
    {"role": "system", "content": input}
]

inputs= tokenizer.apply_chat_template(
    messages, Tokenize= True, add_generation_prompt= True, return_tensors= "pt"
).to("cuda")

In [ ]:
output= model.generate(input_ids= inputs, max_new_tokens= 64, use_cache= True, temperature= 1, min_p= 0.1)
decoded_output= tokenizer.batch_decode(output)

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(decoded_output[0])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 16 Aug 2026

You are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)<|eot_id|><|start_header_id|>system<|end_header_id|>

We are looking to buy licence for our entire engineering department.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

SALES_TEAM<|eot_id|>


In [ ]:
test_data = [
    ("I'd like to know whether you have a yearly payment option.", "SALES_TEAM"),
    ("Our company needs 200 licenses. Who should I contact?", "SALES_TEAM"),
    ("I cannot access my account because the login keeps failing.", "SUPPORT_TEAM"),
    ("The money was deducted but my subscription hasn't started.", "SUPPORT_TEAM"),
    ("There is an extra transaction on my credit card that I don't recognize.", "BILLING_TEAM"),
    ("Please send me the invoice for last month's payment.", "BILLING_TEAM"),
    ("Our API stopped responding after we changed the configuration.", "TECHNICAL_TEAM"),
    ("The dashboard keeps freezing whenever I try to open it.", "TECHNICAL_TEAM"),
    ("I need to add five new employees to our workspace.", "ACCOUNT_TEAM"),
    ("Can I give administrator privileges to another employee?", "ACCOUNT_TEAM"),
    ("What would be the best subscription for a company with 30 employees?", "SALES_TEAM"),
    ("My password reset email never arrived.", "SUPPORT_TEAM"),
    ("Why does my invoice show the wrong tax amount?", "BILLING_TEAM"),
    ("The webhook from your platform is not reaching our server.", "TECHNICAL_TEAM"),
    ("I want to remove an employee who has left our company.", "ACCOUNT_TEAM"),
    ("Do you provide special rates for large organizations?", "SALES_TEAM"),
    ("I was billed twice this month for my subscription.", "BILLING_TEAM"),
    ("The application crashes whenever I upload a file.", "TECHNICAL_TEAM"),
    ("How can I change the owner of our company workspace?", "ACCOUNT_TEAM"),
    ("My payment was declined even though my card is working.", "BILLING_TEAM"),
    ("Can you explain what features come with the enterprise subscription?", "SALES_TEAM"),
    ("I need to recover access to my locked account.", "SUPPORT_TEAM"),
    ("Our integration with the CRM is no longer syncing.", "TECHNICAL_TEAM"),
    ("How do I download all of my previous payment receipts?", "BILLING_TEAM"),
    ("Can I create different permission levels for my team members?", "ACCOUNT_TEAM")
]

In [ ]:
y_true= []
y_pred= []

FastLanguageModel.for_inference(model)

print("Running Evaluation....")

print(f"{'EXPECTED': <20} | {'PREDICTED': <20} | {'MATCH?'}")
print("-"*60)

for query, expected_label in test_data:
  messages= [
    {"role": "system", "content":"You are an intelligent routing assistant. Classify the query into the correct department. Output only the label (Eg: SUPPORT_TEAM)"},
    {"role": "system", "content": query}
]
  inputs= tokenizer.apply_chat_template(messages, tokenize= True, add_generation_prompt= True, return_tensors= "pt").to("cuda")

  output= model.generate(input_ids= inputs, max_new_tokens= 64, use_cache= True, temperature= 1, min_p= 0.1)

  prediction= tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens= True)

  print(f"{expected_label: <20} | {repr(prediction): <20} | {expected_label == prediction.strip()}")



Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Running Evaluation....
EXPECTED             | PREDICTED            | MATCH?
------------------------------------------------------------


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


SALES_TEAM           | 'BILLING_TEAM'       | False


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

SUPPORT_TEAM         | 'SUPPORT_TEAM'       | True


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TECHNICAL_TEAM       | 'TECHNICAL_TEAM'     | True


Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BILLING_TEAM         | 'BILLING_TEAM'       | True
ACCOUNT_TEAM         | 'ACCOUNT_TEAM'       | True


In [ ]:
# Pushing into HuggingFace
from huggingface_hub import login
login("hf_*")

In [ ]:
# Pushed as 16bit Model with Tokenizer
model.push_to_hub_merged("Thamo31/Llama-3.2-3B-4bit-classification-routing-merged", tokenizer, save_method= "merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in Thamo31/Llama-3.2-3B-4bit-classification-routing-merged/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          |  609kB / 1.46GB            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:47<00:00, 113.67s/it]


Unsloth: Merge process complete. Saved to `/content/Thamo31/Llama-3.2-3B-4bit-classification-routing-merged`


In [ ]:
# Pushed as 4bit gguf model
model.push_to_hub_gguf("Thamo31/Llama-3.2-3B-4bit-classification-routing-gguf", tokenizer, quantization_method= "q4_k_m")

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_onv4q2fz/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...2-3B-Instruct.Q4_K_M.gguf:   0%|          | 1.81MB / 2.02GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/Thamo31/Llama-3.2-3B-4bit-classification-routing-gguf
Unsloth: Cleaning up temporary files...


'Thamo31/Llama-3.2-3B-4bit-classification-routing-gguf'